# L17: LLM基础与提示工程


# L17: LLM基础与提示工程

**课时**: 2 课时（90 分钟/课时）

## 1. 学习目标

完成本课程后，学员能够：
1. 解释大语言模型（LLM）的工作原理：自回归生成、注意力机制、Transformer架构
2. 比较GPT、LLaMA、Claude等主流模型的技术路线与特点
3. 掌握提示工程（Prompt Engineering）的核心技巧：零样本、少样本、思维链
4. 设计有效的提示词结构，实现稳定的模型输出控制
5. 了解LLM的能力边界与固有缺陷（幻觉、上下文限制、推理能力）

## 2. 核心知识点

### 2.1 大语言模型概述

**什么是LLM**: 参数量巨大（通常 > 7B），在大规模文本语料上预训练的Transformer decoder模型，能够执行广泛的自然语言任务。

**发展时间线**:
| 年份 | 模型 | 参数量 | 关键突破 |
|------|------|--------|----------|
| 2020 | GPT-3 | 175B | 涌现上下文学习能力 |
| 2022 | ChatGPT | 175B | RLHF对齐人类意图 |
| 2023 | GPT-4 | ~1T (估计) | 多模态、长上下文 |
| 2023 | LLaMA | 7B-65B | 开源羊驼系列基础 |
| 2024 | Claude 3 | - | 长上下文、无害性 |
| 2024 | GPT-4o | - | 原生多模态、实时推理 |

**核心能力**:
- **上下文学习 (In-Context Learning)**: 无需梯度更新，通过提示中的示例学习新任务
- **思维链 (Chain-of-Thought)**: 引导模型展示推理过程，提升复杂任务表现
- **涌现能力 (Emergent Abilities)**: 参数量突破某阈值后突然出现的新能力

### 2.2 GPT系列模型原理

**架构**: Transformer Decoder-only（仅解码器）
- 自回归生成：每次预测下一个token
- 因果注意力掩码：只能看到当前位置之前的上下文

**预训练目标**: 下一个token预测（Next Token Prediction）


In [ ]:
给定序列 x₁, x₂, ..., xₙ
最大化 Σ log P(xₜ | x₁, ..., xₜ₋₁; θ)


**RLHF (Reinforcement Learning from Human Feedback)**:
1. 监督微调（SFT）：用人类标注的对话数据微调
2. 奖励模型（RM）：训练一个模型预测人类偏好
3. PPO强化学习：用RM的奖励信号进一步优化策略


In [ ]:
# 伪代码：GPT自回归生成
def generate(model, prompt, max_tokens=100, temperature=0.7):
    input_ids = tokenize(prompt)
    
    for _ in range(max_tokens):
        # 前向传播
        logits = model(input_ids)
        next_token_logits = logits[:, -1, :]  # 取最后一个token的logits
        
        # 温度采样
        probs = softmax(next_token_logits / temperature)
        next_token = sample(probs)  # 或 greedy argmax
        
        # 追加到序列
        input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)
        
        if next_token == EOS_token:
            break
    
    return detokenize(input_ids)


### 2.3 主流模型对比

| 模型 | 开发商 | 参数量 | 开源 | 特点 |
|------|--------|--------|------|------|
| GPT-4 | OpenAI | ~1T | 否 | 多模态、长上下文、最强推理 |
| GPT-3.5 | OpenAI | 175B | 否 | 轻量、速度快、效果好 |
| Claude 3 | Anthropic | - | 否 | 安全无害、长上下文 |
| LLaMA 3 | Meta | 8B-70B | 是 | 开源可商用、性能接近GPT4 |
| Mistral | Mistral AI | 7B-47B | 是 | 高效率、MoE架构 |
| Qwen | 阿里 | 7B-72B | 是 | 中文优化、开源 |
| DeepSeek | 深度求索 | 7B-67B | 是 | 推理能力强、开源 |

### 2.4 提示工程（Prompt Engineering）

**核心原则**: 清晰、具体、少废话

**零样本提示 (Zero-shot)**:


In [ ]:
将下面的句子改写成正式版本：
输入：我想问一下，那个事情办得咋样了
输出：


→ 直接给出任务描述，不提供示例

**少样本提示 (Few-shot)**:


In [ ]:
将句子改写成正式版本。以下是示例：

输入：我想问一下  输出：请问该事项进展如何
输入：那个东西多少钱  输出：该商品价格是多少

现在请改写：
输入：啥时候能好啊
输出：


→ 提供2-3个示例帮助模型理解任务

**思维链提示 (Chain-of-Thought)**:


In [ ]:
问题：小明有10个苹果，小红给了小明5个，小明吃掉了3个，请问小明现在有多少个苹果？

让我们一步步思考：
1. 小明开始有10个苹果
2. 小红给了小明5个，所以是 10 + 5 = 15 个
3. 小明吃掉了3个，所以是 15 - 3 = 12 个
答案是：12个


**结构化提示词模板**:


In [ ]:
prompt = """
# 角色
你是一位资深数据分析专家，擅长用Python进行数据处理和可视化。

# 任务
分析给定的CSV文件，找出销售额最高的前10个产品，并绘制柱状图。

# 输出要求
1. 读取数据并显示前5行
2. 按产品分组计算总销售额
3. 排序并取Top 10
4. 使用matplotlib绘制柱状图
5. 保存图片到当前目录

# 约束
- 使用pandas和matplotlib
- 代码需要完整可运行
- 添加必要的注释

# 数据文件路径
/data/sales.csv
"""


### 2.5 LLM的局限性

**幻觉（Hallucination）**: 模型生成看似合理但实际错误的内容
- 原因：概率生成、训练数据偏差、缺乏真实验证
- 缓解：检索增强（RAG）、事实核查提示、结构化输出

**上下文长度限制**: 模型能处理的token数量有上限
- 解决：分块处理、摘要压缩、滑动窗口

**推理能力有限**: 复杂逻辑推理、多步计算仍然困难
- 解决：思维链提示、拆解任务步骤、使用外部工具

## 3. 代码示例


In [ ]:
"""
L17-prompt-engineering.py
演示：提示工程核心技巧与LLM调用
"""

# ============ 1. 使用 OpenAI API 调用 GPT ============
import os

# 设置API密钥（请替换为你的密钥）
# os.environ["OPENAI_API_KEY"] = "your-api-key"

# 如果没有API密钥，可以使用开源模型或模拟
# 以下代码展示提示工程的四种核心技巧

# ============ 2. 零样本提示 ============
zero_shot_prompt = """
请判断以下评论的情感是正面、负面还是中性：

评论：这家餐厅的菜品非常一般，服务也很慢。
情感：
"""

# ============ 3. 少样本提示 ============
few_shot_prompt = """
请将中文俗语翻译成英文。以下是翻译示例：

中文：画蛇添足  英文：To add legs to a snake (unnecessary extra work)
中文：揠苗助长  英文：To pull up seedlings to help them grow (counterproductive help)
中文：掩耳盗铃  英文：To cover one's ears while stealing a bell (self-deception)

请翻译：
中文：亡羊补牢  英文：
"""

# ============ 4. 思维链提示 ============
cot_prompt = """
问题：如果一个人买了一件外套花了150元，又买了一双鞋子花了200元，
然后退掉了外套，他实际上花了多少钱？

让我们一步步推理：
"""

# ============ 5. 结构化输出提示 ============
structured_prompt = """
你是一个图书信息提取助手。请从以下文本中提取信息，并以JSON格式输出。

文本：「《深度学习》由Ian Goodfellow等人撰写，2016年由MIT出版社出版，共800页。」

请提取：书名、作者、出版年份、出版社、页数

输出格式（严格遵循）：
{
    "title": "",
    "authors": [],
    "publish_year": null,
    "publisher": "",
    "pages": null
}
"""

# ============ 6. 对比实验函数 ============
def compare_prompts(prompts, descriptions):
    """对比不同提示策略的效果"""
    results = []
    for prompt, desc in zip(prompts, descriptions):
        print(f"\n{'='*50}")
        print(f"策略: {desc}")
        print(f"提示内容: {prompt[:100]}...")
        print("-" * 50)
        # 这里应该调用API，为了演示打印提示
        results.append({"description": desc, "prompt": prompt})
    return results

prompts = [zero_shot_prompt, few_shot_prompt, cot_prompt, structured_prompt]
descriptions = ["零样本", "少样本", "思维链", "结构化输出"]
results = compare_prompts(prompts, descriptions)

print("\n提示工程对比实验完成！")
print("提示：实际使用时需要调用API（如OpenAI、Anthropic或本地模型）")


## 4. 练习题

1. **原理解释**: 画图说明GPT的自回归生成过程，并解释为什么Decoder-only架构适合语言建模。
2. **模型对比**: 比较GPT-4、Claude、LLaMA三种模型的技术路线差异，分析各自的优势和适用场景。
3. **提示设计**: 为以下任务设计提示词（零样本、少样本、思维链各一种）：
   - 将一段文字压缩成50字以内的摘要
   - 判断一段代码是否有安全漏洞
   - 从简历文本中提取关键信息（姓名、学历、工作年限）
4. **幻觉分析**: 分析以下场景中幻觉产生的原因，并提出缓解方案：
   - 模型生成了一个不存在的论文引用
   - 模型给出了错误的中文翻译
5. **实践应用**: 选择一个你感兴趣的实际任务（如写作辅助、数据分析、代码生成），设计一套完整的提示词模板，并在至少3个不同输入上测试效果。

## 5. 延伸阅读

- **论文**: Brown et al., 2020, "Language Models are Few-Shot Learners" — GPT-3论文，上下文学习的基础
- **论文**: Ouyang et al., 2022, "Training language models to follow instructions with human feedback" — InstructGPT/ChatGPT基础
- **网站**: Prompt Engineering Guide — https://www.promptingguide.ai/
- **网站**: OpenAI Prompt Examples — https://platform.openai.com/examples
- **工具**: Anthropic Claude API — https://docs.anthropic.com/
- **工具**: LLaMA.cpp — 本地运行开源LLM https://github.com/ggerganov/llama.cpp

---
